In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import utilities.plot_settings
from matplotlib import rc
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

rc("text", usetex=True)
rc("font", family="serif")
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
yellow = plt.cm.Set2(5 / 7)
orange = plt.cm.Set2(1 / 7)
sea = plt.cm.Set2(0 / 7)
blue = plt.cm.Set2(2 / 7)
green = plt.cm.Set2(4 / 7)
purple = plt.cm.Set2(3 / 7)


In [ ]:
SMALL_SIZE = 20
MEDIUM_SIZE = 28
BIGGER_SIZE = 36

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=BIGGER_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE) 

In [ ]:
def pdf_log10_magnetic_field_2normal(log10B: np.ndarray, mean_2,sigma_2,w) -> np.ndarray:
    """
    Mixture of two Gaussian distributions for the logarithm log10 of neutron stars' initial magnetic field.
    The means and standard deviations are defined in the configuration file.

    Args:
        log10B (np.ndarray): log10 of the magnetic field strength in [G].

    Returns:
        (np.ndarray): Value of the pdf for each log10B.
    """

    # Define the means and dispersions of the two Gaussian components.
    mean_1 = 13.09
    sigma_1 = 0.5

    # Define the fractional contribution of the first Gaussian.
    if (w < 0) or (w > 1):
        raise ValueError(
            "The relative weight parameter of the double_log-normal initial magnetic field model "
            "must be in the range 0 and 1."
        )

    pdf_gaussian_1 = (
        1.0
        / (np.sqrt(2 * np.pi) * sigma_1)
        * np.exp(-((log10B - mean_1) ** 2) / (2.0 * sigma_1**2))
    )

    pdf_gaussian_2 = (
        1.0
        / (np.sqrt(2 * np.pi) * sigma_2)
        * np.exp(-((log10B - mean_2) ** 2) / (2.0 * sigma_2**2))
    )

    pdf = w * pdf_gaussian_1 + (1.0 - w) * pdf_gaussian_2

    return pdf

In [ ]:
def pdf_gaussian_custom_norm(
    x: np.ndarray, mean: float, sigma: float, norm: float
) -> np.ndarray:
    """
    A Gaussian function with custom normalization.
    This function is needed in the pdf_log10_magnetic_field_smooth_tophat function where the normalization
    of the two Gaussian components is regulated by the slope of the central region (see implementation of
    function pdf_log10_magnetic_field_smooth_tophat).

    Args:
        x (np.ndarray): Array of values where to compute the function.
        mean (float): Mean of the Gaussian distribution.
        sigma (float): Standard deviation of the Gaussian distribution.
        norm (float): Normalization of the Gaussian distribution.

    Returns:
        (np.ndarray): Value of the Gaussian distribution for each x.
    """
    gauss_func = norm * np.exp(-0.5 * ((x - mean) / sigma) ** 2)

    return gauss_func

In [ ]:
def pdf_log10_magnetic_field_smooth_tophat(
    log10B: np.ndarray,
) -> np.ndarray:
    """
    Initial magnetic fields following a smooth top-hat function. This function is composed of a Gaussian rise
    up to the peak of a first Gaussian, a subsequent straight sloped line extending from the first Gaussian peak
    to a second normal component and final Gaussian decay.
    The parameters of this distribution are defined in the configuration file.

    Args:
        log10B (np.ndarray): log10 of the magnetic field strength in [G].

    Returns:
        (np.ndarray): Value of the pdf for each log10B.
    """
    # Define the parameters of the distribution.
    rise_mean = 13.09
    rise_sigma = 0.5
    decay_mean = 14.0
    decay_sigma = 0.2
    slope = 0.0

    # Calculate the x coordinate of the center of the sloped segment.
    center = (rise_mean + decay_mean) / 2

    # Evaluate the heights of the two Gaussians assuming that the sloped straight segment passes through the point with
    # coordinates (center, 1). Since we are only interested in sampling from this distribution, we are not
    # concerned about the exact normalization of this pdf. Therefore, the y-coordinate of this point is chosen
    # for simplicity to be 1 but could be any arbitrary value. In other words, the slope is defined by a line with
    # a given slope passing through the point with coordinates (center, 1) beginning at the peak of the first Gaussian
    # component and ending at the peak of the second Gaussian component.
    norm_rise_gaussian = slope * (rise_mean - center) + 1
    norm_decay_gaussian = slope * (decay_mean - center) + 1

    # Gaussian rise on the left side.
    rise_gaussian = pdf_gaussian_custom_norm(
        log10B, rise_mean, rise_sigma, norm_rise_gaussian
    )

    # Gaussian decay on the right side.
    decay_gaussian = pdf_gaussian_custom_norm(
        log10B, decay_mean, decay_sigma, norm_decay_gaussian
    )

    # Central sloped region.
    central_region = np.where(
        (log10B >= rise_mean) & (log10B <= decay_mean),
        slope * (log10B - center) + 1,
        0,
    )

    # Combine the Gaussian rise, sloped region, and Gaussian decay.
    smooth_top_hat = np.maximum(rise_gaussian, central_region)
    smooth_top_hat = np.maximum(smooth_top_hat, decay_gaussian)

    return smooth_top_hat

In [ ]:

mean_1 = 13.09
mean_2 = 14
sigma_1 = 0.5
sigma_2 = 0.5

# Define the fractional contribution of the first Gaussian.
w = 0.5
x = np.linspace(10, 17, 500)
y1_0_5 = pdf_log10_magnetic_field_2normal(x, mean_2,sigma_2,w)

In [ ]:

plt.figure(facecolor='none')  # Transparent figure

# Example plots
plt.title('$B_{0,\\text{ magnetar}} =  10^{14}$ G')
plt.plot(x, y1_0_5, color=sea, linewidth=2)#, label=)
# plt.plot(x, y1_0_6, color='green', linewidth=2, label='Magnetar fraction = 0.6')
# --- Add label at the peak of the second (rightmost) peak ---
# Find local maxima (peaks) without SciPy
plt.xlim(11,17)
#plt.ylim(0,0.6)

plt.xlabel(r'$B_{\rm init} \,[\mathrm{G}]$')

# Custom x-ticks
plt.xticks([11, 13,  15, 17], [r'$10^{11}$',r'$10^{13}$', r'$10^{15}$',r'$10^{17}$'])
#plt.yticks([])

# Transparent axes background
plt.gca().set_facecolor('none')

# Add legend at best location
plt.legend(loc='best', frameon=False)

plt.savefig('/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/initial_magnetic_field.pdf', transparent=True)
plt.show()


In [ ]:
# Physical magnetar B0 means in Gauss
means_B0 = [7.5e13, 1e14, 2.5e14, 5e14, 7.5e14, 1e15]

# Convert to log10 since the PDF uses log-space means
means_log10 = np.log10(means_B0)

# Custom labels
B_0_labels = [
    '$ 7.5 \\times 10^{13}$ G',
    '$ 10^{14}$ G',
    '$ 2.5 \\times 10^{14}$ G',
    '$ 5 \\times 10^{14}$ G',
    '$ 7.5 \\times 10^{14}$ G',
    '$ 10^{15}$ G'
]

# Colors
colors = plt.cm.viridis(np.linspace(0, 1, len(means_log10)))

plt.figure(facecolor='none')

for i, mean_2 in enumerate(means_log10):
    y = pdf_log10_magnetic_field_2normal(x, mean_2, sigma_2, w)
    plt.plot(x, y, color=colors[i], linewidth=2, label=B_0_labels[i])

plt.xlim(11, 17)
plt.xlabel(r'$B_{\rm init}\,[\mathrm{G}]$')
plt.xticks([11, 13, 15, 17],
           [r'$10^{11}$', r'$10^{13}$', r'$10^{15}$', r'$10^{17}$'])
#plt.yticks([])

plt.gca().set_facecolor('none')

plt.legend(title='$B_{0, \\rm magnetars}$',
           frameon=False, fontsize=15, loc='upper left')
plt.savefig('/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/initial_magnetic_field.pdf', transparent=False)

plt.tight_layout()
plt.show()
